<a href="https://colab.research.google.com/github/alottamyles/Hystrix/blob/2026fall_myles/coding_exercise_ML_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# Data source: Kaggle - USA House Sales Data
# URL: https://www.kaggle.com/datasets/abdulwadood11220/usa-house-sales-data
# Read dataset
df = pd.read_csv('us_house_Sales_data.csv')
# Convert text values into numbers for Price, Area (Sqft), and Location
df["Price"] = df["Price"].str.replace("$", "").str.replace(",", "").astype(float)
df["Area (Sqft)"] = df["Area (Sqft)"].str.replace(" sqft", "").astype(float)
#Rename columns to match dataset
df = df.rename(columns={
    "Price": "price",
    "Area (Sqft)": "square_footage",
    "City": "location"
})
# Features and target
X = df[['square_footage', 'location']]
y = df['price']
# Preprocessing: One-hot encode the location column
preprocessor = ColumnTransformer(
transformers=[
('location', OneHotEncoder(sparse_output=False), ['location'])
], remainder='passthrough')
# Create pipeline with preprocessing and model
model = Pipeline(steps=[
('preprocessor', preprocessor),
('regressor', LinearRegression())
])
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)
# Train model
model.fit(X_train, y_train)
# Make prediction for a new house: 2000 sq ft in Los Angeles
new_house = pd.DataFrame({'square_footage': [2000], 'location': ['Los Angeles']})
predicted_price = model.predict(new_house)
print(f"Predicted price for a 2000 sq ft house in Los Angeles: ${predicted_price[0]:,.2f}")
# Display model coefficients

# Square_footage coefficient represents the estimated change in house price for each additional sqft
# Location coefficient represents the connection between city and price, taking square footage into account
# Positive (+) values = higher predicted prices
# Negative (-) values = lower predicted prices
feature_names = (
    model.named_steps['preprocessor']
    .named_transformers_['location']
    .get_feature_names_out(['location'])
    .tolist()
    + ['square_footage']
)
coefficients = model.named_steps['regressor'].coef_
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

Predicted price for a 2000 sq ft house in Los Angeles: $827,269.28

Model Coefficients:
location_Fresno: 17640.38
location_Los Angeles: 7741.22
location_Sacramento: -7750.17
location_San Diego: -947.34
location_San Francisco: -16684.09
square_footage: -18.42
